# Limpieza de datos - turno 2: Ricardo

### De acuerdo con el [plan de limpieza](../docs/plan_limpieza.md), este cuaderno ejecuta las reglas aprobadas. Los posibles duplicados se revisan y documentan sin eliminarlos automáticamente.

### Esta limpieza forma parte del proceso reproducible que posteriormente valida el conjunto final.

## 1. Preparación reproducible

El módulo `src/limpieza.py` concentra las funciones de limpieza del TURNO 2. El notebook únicamente las ejecuta sobre el DataFrame crudo (`cargar_datos()`) y presenta el resultado: no hay ninguna transformación escrita directamente aquí.

Cada variable corregida (`DIRECCION`, `TELEFONO`, `SUPERVISOR`, `DIRECTOR`) conserva su valor crudo en una columna `_ORIGINAL`, y las reglas que solo marcan posibles duplicados o imputaciones agregan columnas nuevas en vez de sobrescribir sin rastro. Nada se elimina.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.diagnostico import cargar_datos
from src.limpieza import limpiar_datos_preliminar, revisar_duplicados_parciales

df = cargar_datos()
limpio_pre = limpiar_datos_preliminar(df)

print(f"Registros: {len(limpio_pre):,}")
print(f"Columnas originales: {len(df.columns)} -> columnas tras limpieza preliminar: {len(limpio_pre.columns)}")


Registros: 11,890
Columnas originales: 19 -> columnas tras limpieza preliminar: 52


## 2. `ESTABLECIMIENTO`

Se conserva el nombre visible. Los pares parciales se comparan con RapidFuzz dentro del mismo municipio, dirección, jornada y plan. Cada par recibe una decisión documentada y no se elimina automáticamente.

In [2]:
revisados = revisar_duplicados_parciales(limpio_pre)
n_pares = len(revisados)
n_conservados = int(revisados['decision'].eq('CONSERVAR').sum())
n_pendientes = int(revisados['decision'].eq('PENDIENTE').sum())

resumen_establecimiento = pd.DataFrame([
    {'indicador': 'Pares candidatos revisados con RapidFuzz', 'cantidad': n_pares},
    {'indicador': 'Pares conservados por códigos MINEDUC distintos', 'cantidad': n_conservados},
    {'indicador': 'Decisiones pendientes', 'cantidad': n_pendientes},
])
display(resumen_establecimiento)
display(Markdown('**Ejemplos de pares revisados:**'))
display(revisados.head(10))
display(Markdown(
    f'**Conclusión:** se revisaron {n_pares:,} pares con similitud mínima de 95. '
    f'Los {n_conservados:,} se conservan porque poseen códigos MINEDUC distintos y no existe evidencia oficial '
    f'para fusionarlos o eliminar uno. Quedan {n_pendientes:,} decisiones pendientes.'
))


,indicador,cantidad
0,Pares candidatos revisados con RapidFuzz,1085
1,Pares conservados por códigos MINEDUC distintos,1085
2,Decisiones pendientes,0


**Ejemplos de pares revisados:**

,par_id,codigo_1,codigo_2,municipio,establecimiento_1,establecimiento_2,similitud_nombre,direccion,jornada,plan,decision,justificacion
0,1,13-27-0068-46,13-27-7032-46,Aguacatán,"COLEGIO PRIVADO ""EL NUEVO MILLENIUM""",COLEGIO PRIVADO EL NUEVO MILLENIUM,100.00,"3A. AVENIDA 3-01, ZONA 1",VESPERTINA,DIARIO(REGULAR),CONSERVAR,Los códigos MINEDUC son distintos y representa...
1,2,13-27-0066-46,13-27-0085-46,Aguacatán,INSTITUTO MAYANCE AGUACATECO,INSTITUTO MAYANCE AGUACATECO,100.00,4TA CALLE 3-35 ZONA 3,VESPERTINA,DIARIO(REGULAR),CONSERVAR,Los códigos MINEDUC son distintos y representa...
2,3,01-14-0213-46,01-14-0220-46,Amatitlán,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,100.00,1RA Y 2DA CALLE LOTE 33 A COLONIA SAN JUAN BAU...,VESPERTINA,DIARIO(REGULAR),CONSERVAR,Los códigos MINEDUC son distintos y representa...
3,4,01-14-0016-46,01-14-0018-46,Amatitlán,INSTITUTO PRIVADO MIXTO DE COMPUTACION INFORMA...,INSTITUTO PRIVADO MIXTO DE COMPUTACION INFORMA...,96.15,"2A. AVENIDA, 12-44 CANTON SAN ANTONIO",DOBLE,FIN DE SEMANA,CONSERVAR,Los códigos MINEDUC son distintos y representa...
4,5,01-14-0141-46,01-14-8051-46,Amatitlán,"LICEO MIXTO CRISTIANO ""JIREH""",LICEO MIXTO CRISTIANO 'JIREH',100.00,"3A. AVENIDA 2-45, COLONIA VILLA ALBORADA",DOBLE,FIN DE SEMANA,CONSERVAR,Los códigos MINEDUC son distintos y representa...
5,6,01-14-0135-46,01-14-0136-46,Amatitlán,COLEGIO PEDAGOGICO INTEGRAL ALBORADA,COLEGIO PEDAGOGICO INTEGRAL ALBORADA,100.00,4 AVENIDA 10-57,VESPERTINA,DIARIO(REGULAR),CONSERVAR,Los códigos MINEDUC son distintos y representa...
6,7,01-14-0050-46,01-14-0052-46,Amatitlán,COLEGIO PEDAGOGICO INTEGRAL ALBORADA,COLEGIO PEDAGOGICO INTEGRAL ALBORADA,100.00,"4A. AVENIDA 10-57, CANTON EL ROSARIO",DOBLE,FIN DE SEMANA,CONSERVAR,Los códigos MINEDUC son distintos y representa...
7,8,01-14-0058-46,01-14-0189-46,Amatitlán,COLEGIO MIXTO CULTURA,COLEGIO MIXTO CULTURA,100.00,4A. AVENIDA 5-34,VESPERTINA,DIARIO(REGULAR),CONSERVAR,Los códigos MINEDUC son distintos y representa...
8,9,01-14-0225-46,01-14-0226-46,Amatitlán,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,100.00,5A. AVENIDA FINAL BARRIO EL INGENIO,NOCTURNA,DIARIO(REGULAR),CONSERVAR,Los códigos MINEDUC son distintos y representa...
9,10,01-14-0082-46,01-14-0083-46,Amatitlán,LICEO MIXTO AMATITLANECO,LICEO MIXTO AMATITLANECO,100.00,"5A. CALLE, 3-73, BARRIO LA CRUZ",DOBLE,FIN DE SEMANA,CONSERVAR,Los códigos MINEDUC son distintos y representa...


**Conclusión:** se revisaron 1,085 pares con similitud mínima de 95. Los 1,085 se conservan porque poseen códigos MINEDUC distintos y no existe evidencia oficial para fusionarlos o eliminar uno. Quedan 0 decisiones pendientes.

## 3. `DIRECCION`

Se aplican en cascada las 4 reglas del plan: faltante disfrazado a NA, recorte del sufijo de municipio redundante, recorte de fecha incrustada al final, y corrección de la letra "O" por "0" en contexto numérico. El dominio rural no se penaliza por no tener número de casa.

In [3]:
faltante_na = int(limpio_pre["DIRECCION"].isna().sum())
faltante_crudo = int(df["DIRECCION"].fillna("").str.strip().eq("").sum())

cambio_texto = limpio_pre["DIRECCION_ORIGINAL"].fillna("") != limpio_pre["DIRECCION"].fillna("")
n_modificadas = int(cambio_texto.sum())

resumen_direccion = pd.DataFrame([
    {"regla": "Faltante disfrazado -> NA (vacía o igual al municipio)", "cantidad": faltante_na},
    {"regla": "Cualquier cambio de texto (municipio/fecha/O-por-cero)", "cantidad": n_modificadas},
])
display(resumen_direccion)

ejemplos = limpio_pre.loc[cambio_texto, ["DIRECCION_ORIGINAL", "DIRECCION"]].drop_duplicates().head(6)
display(Markdown("**Ejemplos antes / después:**"))
display(ejemplos)

display(Markdown(
    f"**Conclusión:** {faltante_na:,} direcciones quedaron `NA` por ser vacías o repetir exactamente el "
    f"municipio (el conteo crudo de vacías era {faltante_crudo:,}; la diferencia son las que solo decían el "
    f"nombre del municipio). Otras {n_modificadas:,} celdas cambiaron de texto: se recortó el sufijo de "
    "municipio redundante, la fecha incrustada al final, o se corrigió la letra “O” por “0” en contexto "
    "numérico. El valor original completo queda en `DIRECCION_ORIGINAL` para poder auditar cada cambio."
))


,regla,cantidad
0,Faltante disfrazado -> NA (vacía o igual al mu...,271
1,Cualquier cambio de texto (municipio/fecha/O-p...,1278


**Ejemplos antes / después:**

,DIRECCION_ORIGINAL,DIRECCION
8,7A. CALLE 11-09 ZONA 6 COBAN,7A. CALLE 11-09 ZONA 6
59,"DIAGONAL 1 1-26, ZONA 1 COBÁN","DIAGONAL 1 1-26, ZONA 1"
65,2A. CALLE 1-99 ZONA 4 COBAN,2A. CALLE 1-99 ZONA 4
68,"2A. CALLE 8-18 ZONA 4, COBAN",2A. CALLE 8-18 ZONA 4
90,6A. CALLE 2-06 ZONA 8 PERIFERICO SUR COBAN,6A. CALLE 2-06 ZONA 8 PERIFERICO SUR
102,COBAN,<NA>


**Conclusión:** 271 direcciones quedaron `NA` por ser vacías o repetir exactamente el municipio (el conteo crudo de vacías era 99; la diferencia son las que solo decían el nombre del municipio). Otras 1,278 celdas cambiaron de texto: se recortó el sufijo de municipio redundante, la fecha incrustada al final, o se corrigió la letra “O” por “0” en contexto numérico. El valor original completo queda en `DIRECCION_ORIGINAL` para poder auditar cada cambio.

## 4. `TELEFONO`

Vacío queda como `NA`. El resto se procesa con `separar_numeros()` y se representa como una lista de contactos de ocho dígitos. Los números de siete dígitos no se completan por inferencia.

In [4]:
telefono_vacio_crudo = int(df['TELEFONO'].fillna('').str.strip().eq('').sum())
telefono_na = int(limpio_pre['TELEFONO'].isna().sum())
telefono_legado = int(limpio_pre['TELEFONO_DESCARTO_7_DIGITOS'].sum())
con_varios = int(limpio_pre['TELEFONO'].str.contains(';', na=False).sum())

resumen_telefono = pd.DataFrame([
    {'indicador': 'Vacío en el original', 'cantidad': telefono_vacio_crudo},
    {'indicador': 'NA tras la limpieza', 'cantidad': telefono_na},
    {'indicador': 'Celdas con algún número legado descartado', 'cantidad': telefono_legado},
    {'indicador': 'Celdas con más de un número válido', 'cantidad': con_varios},
])
display(resumen_telefono)

ejemplos_tel = limpio_pre.loc[
    limpio_pre['TELEFONO_ORIGINAL'].fillna('').ne('')
    & (limpio_pre['TELEFONO_ORIGINAL'] != limpio_pre['TELEFONO']),
    ['TELEFONO_ORIGINAL', 'TELEFONO'],
].drop_duplicates().head(6)
display(Markdown('**Ejemplos antes / después:**'))
display(ejemplos_tel)
display(Markdown(
    f'**Conclusión:** el formato final conserva únicamente contactos de ocho dígitos. '
    f'En {telefono_legado:,} celdas se descartó algún componente de siete dígitos sin inventar información; '
    f'{telefono_na:,} celdas quedan como `NA`. `TELEFONO_ORIGINAL` conserva el texto crudo.'
))


,indicador,cantidad
0,Vacío en el original,969
1,NA tras la limpieza,1078
2,Celdas con algún número legado descartado,90
3,Celdas con más de un número válido,135


**Ejemplos antes / después:**

,TELEFONO_ORIGINAL,TELEFONO
326,78208583-78209143,78208583; 78209143
484,79540830-79540909,79540830; 79540909
655,78391288-78392217,78391288; 78392217
674,79649696-78739432,79649696; 78739432
676,78739432-79649696,78739432; 79649696
772,78393245-78393246,78393245; 78393246


**Conclusión:** el formato final conserva únicamente contactos de ocho dígitos. En 90 celdas se descartó algún componente de siete dígitos sin inventar información; 1,078 celdas quedan como `NA`. `TELEFONO_ORIGINAL` conserva el texto crudo.

## 5. `SUPERVISOR`

Se corrigen grafías puntuales y variantes dentro del mismo distrito. La ausencia real queda como `NA`; no se imputa porque el conjunto no contiene fecha de vigencia.

In [5]:
sup_ausente_final = int(limpio_pre['SUPERVISOR'].isna().sum())
sup_imputado = int(limpio_pre['SUPERVISOR_IMPUTADO'].sum())

resumen_supervisor = pd.DataFrame([
    {'indicador': 'Sin supervisor identificable', 'cantidad': sup_ausente_final},
    {'indicador': 'Valores imputados (debe ser 0)', 'cantidad': sup_imputado},
])
display(resumen_supervisor)

ejemplos_sup = limpio_pre.loc[
    limpio_pre['SUPERVISOR_ORIGINAL'].fillna('').ne('')
    & limpio_pre['SUPERVISOR'].notna()
    & (limpio_pre['SUPERVISOR_ORIGINAL'] != limpio_pre['SUPERVISOR']),
    ['SUPERVISOR_ORIGINAL', 'SUPERVISOR'],
].drop_duplicates().head(6)
display(Markdown('**Ejemplos de corrección de grafía o variantes:**'))
display(ejemplos_sup)
display(Markdown(
    f'**Conclusión:** {sup_ausente_final:,} filas quedan como `NA` y el número de imputaciones es {sup_imputado:,}. '
    'No se completa un supervisor sin una referencia institucional con vigencia.'
))


,indicador,cantidad
0,Sin supervisor identificable,561
1,Valores imputados (debe ser 0),0


**Ejemplos de corrección de grafía o variantes:**

,SUPERVISOR_ORIGINAL,SUPERVISOR
930,"AMALIA ESTER LIX SOCOP DE CHUY,",AMALIA ESTER LIX SOCOP DE CHUY
1225,EDNA ODILIA ACEVED0,EDNA ODILIA ACEVEDO
1349,CELESTINA ARLETTY HERNÁNDEZ AGUILAR,CELESTINA ARLETTY HERNÁNDEZ AGUILAR
1375,AMADO SALOMON FLORES PEREZ,AMADO SALOMON FLORES PEREZ
3493,MARTA ELIDA CARIAS HERNANDEZ DE GARCIA,MARTA ELIDA CARIAS HERNANDEZ DE GARCIA
3517,EDGAR ROLANDO JUÁREZ ORTÌZ,EDGAR ROLANDO JUÁREZ ORTÍZ


**Conclusión:** 561 filas quedan como `NA` y el número de imputaciones es 0. No se completa un supervisor sin una referencia institucional con vigencia.

## 6. `DIRECTOR`

Se separa el título incrustado, se reclasifica a NA la ausencia real (incluye los disfrazados) y se fusionan variantes dentro del mismo MUNICIPIO.

In [6]:
dir_titulo = int(limpio_pre["DIRECTOR_TITULO"].notna().sum())
dir_ausente_final = int(limpio_pre["DIRECTOR"].isna().sum())
dir_ausente_crudo = int(df["DIRECTOR"].fillna("").str.strip().eq("").sum())

resumen_director = pd.DataFrame([
    {"indicador": "Títulos separados (LIC./LICDA./PEM.)", "cantidad": dir_titulo},
    {"indicador": "Ausencia real -> NA (vacíos + disfrazados)", "cantidad": dir_ausente_final},
])
display(resumen_director)

ejemplos_titulo = limpio_pre.loc[
    limpio_pre["DIRECTOR_TITULO"].notna(), ["DIRECTOR_ORIGINAL", "DIRECTOR", "DIRECTOR_TITULO"]
].drop_duplicates().head(5)
display(Markdown("**Ejemplos de título separado:**"))
display(ejemplos_titulo)

ejemplos_disfrazado = limpio_pre.loc[
    limpio_pre["DIRECTOR"].isna() & limpio_pre["DIRECTOR_ORIGINAL"].fillna("").str.strip().ne(""),
    "DIRECTOR_ORIGINAL",
].drop_duplicates().head(6)
display(Markdown("**Ejemplos de faltante disfrazado reclasificado a NA:**"))
display(ejemplos_disfrazado)

display(Markdown(
    f"**Conclusión:** se separaron {dir_titulo:,} títulos incrustados a `DIRECTOR_TITULO`, dejando el nombre "
    f"limpio en DIRECTOR. {dir_ausente_final:,} filas ({dir_ausente_final / len(limpio_pre) * 100:.1f}%) "
    f"quedan `NA` (el conteo crudo de vacías era {dir_ausente_crudo:,}: el resto son disfrazados como "
    "guiones, ceros o \"SIN DATOS\", que la regla de \"menos de 2 palabras con letra\" también detecta). No "
    "es un dato imputable: es propio del establecimiento. Las variantes de tildes/espacios se fusionan "
    "dentro del mismo MUNICIPIO -no DISTRITO como en SUPERVISOR- para no mezclar homónimos de otra "
    "jurisdicción."
))


,indicador,cantidad
0,Títulos separados (LIC./LICDA./PEM.),20
1,Ausencia real -> NA (vacíos + disfrazados),2174


**Ejemplos de título separado:**

,DIRECTOR_ORIGINAL,DIRECTOR,DIRECTOR_TITULO
513,LIC. ELGI WALTER BOTEO GARCÍA,ELGI WALTER BOTEO GARCÍA,LIC.
567,PEM. ZOILA CÁNDIDA LUNA PÉREZ,ZOILA CÁNDIDA LUNA PÉREZ,PEM.
711,LIC. CARLOS HUMBERTO CABRERA OVALLE,CARLOS HUMBERTO CABRERA OVALLE,LIC.
2404,LICDA. DORIS ZUNUN CARRERA,DORIS ZUNUN CARRERA,LICDA.
2742,LICDA. LILIA ADRIANA LEMUS GALÁN,LILIA ADRIANA LEMUS GALÁN,LICDA.


**Ejemplos de faltante disfrazado reclasificado a NA:**

0                 --
102              ---
338                -
790                .
1326            ----
1332    ------------
Name: DIRECTOR_ORIGINAL, dtype: string

**Conclusión:** se separaron 20 títulos incrustados a `DIRECTOR_TITULO`, dejando el nombre limpio en DIRECTOR. 2,174 filas (18.3%) quedan `NA` (el conteo crudo de vacías era 1,755: el resto son disfrazados como guiones, ceros o "SIN DATOS", que la regla de "menos de 2 palabras con letra" también detecta). No es un dato imputable: es propio del establecimiento. Las variantes de tildes/espacios se fusionan dentro del mismo MUNICIPIO -no DISTRITO como en SUPERVISOR- para no mezclar homónimos de otra jurisdicción.

## 7. Resumen consolidado de la limpieza TURNO 2

La tabla siguiente reúne las cantidades afectadas por cada regla, recalculadas en cada ejecución para que el número nunca quede desactualizado respecto al código.

In [7]:
telefono_vacio_crudo = int(df['TELEFONO'].fillna('').str.strip().eq('').sum())
telefono_na = int(limpio_pre['TELEFONO'].isna().sum())
telefono_legado = int(limpio_pre['TELEFONO_DESCARTO_7_DIGITOS'].sum())
con_varios = int(limpio_pre['TELEFONO'].str.contains(';', na=False).sum())

resumen_telefono = pd.DataFrame([
    {'indicador': 'Vacío en el original', 'cantidad': telefono_vacio_crudo},
    {'indicador': 'NA tras la limpieza', 'cantidad': telefono_na},
    {'indicador': 'Celdas con algún número legado descartado', 'cantidad': telefono_legado},
    {'indicador': 'Celdas con más de un número válido', 'cantidad': con_varios},
])
display(resumen_telefono)

ejemplos_tel = limpio_pre.loc[
    limpio_pre['TELEFONO_ORIGINAL'].fillna('').ne('')
    & (limpio_pre['TELEFONO_ORIGINAL'] != limpio_pre['TELEFONO']),
    ['TELEFONO_ORIGINAL', 'TELEFONO'],
].drop_duplicates().head(6)
display(Markdown('**Ejemplos antes / después:**'))
display(ejemplos_tel)
display(Markdown(
    f'**Conclusión:** el formato final conserva únicamente contactos de ocho dígitos. '
    f'En {telefono_legado:,} celdas se descartó algún componente de siete dígitos sin inventar información; '
    f'{telefono_na:,} celdas quedan como `NA`. `TELEFONO_ORIGINAL` conserva el texto crudo.'
))


,indicador,cantidad
0,Vacío en el original,969
1,NA tras la limpieza,1078
2,Celdas con algún número legado descartado,90
3,Celdas con más de un número válido,135


**Ejemplos antes / después:**

,TELEFONO_ORIGINAL,TELEFONO
326,78208583-78209143,78208583; 78209143
484,79540830-79540909,79540830; 79540909
655,78391288-78392217,78391288; 78392217
674,79649696-78739432,79649696; 78739432
676,78739432-79649696,78739432; 79649696
772,78393245-78393246,78393245; 78393246


**Conclusión:** el formato final conserva únicamente contactos de ocho dígitos. En 90 celdas se descartó algún componente de siete dígitos sin inventar información; 1,078 celdas quedan como `NA`. `TELEFONO_ORIGINAL` conserva el texto crudo.